# MedSAM2-Guided Retinal Blood Vessel Segmentation


**Author:** Kenny Tran

## Attribution Summary

This notebook combines original code with components adapted from external sources.

**External :**
- MedSAM2 repository (Ma et al., 2024) repo cloned and `build_sam2` API used to load the pretrained image encoder.
- MedSAM2 `MedSAM2_latest.pt` checkpoint
- U-Net architecture (Ronneberger et al., 2015)
- clDice loss (Shit et al., 2021) the topology-preserving loss equation


## Setup & Install MedSAM2




In [1]:
!nvidia-smi

Tue May  5 03:21:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Clone MedSAM2 repository (NOT facebookresearch/sam2)
!git clone https://github.com/bowang-lab/MedSAM2.git
%cd MedSAM2
!pip install -e . -q
%cd ..

Cloning into 'MedSAM2'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 297 (delta 91), reused 64 (delta 64), pack-reused 132 (from 1)
Receiving objects: 100% (297/297), 18.82 MiB | 15.60 MiB/s, done.
Resolving deltas: 100% (122/122), done.
/content/MedSAM2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 14.7 MB/s eta 0:00:00
  Building editable for MedSAM2 (pyproject.toml) ... done
/content


In [3]:
!pip install -q opencv-python scikit-image tqdm huggingface_hub

In [4]:
import os, sys, gzip, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from glob import glob
import cv2
from PIL import Image
from tqdm.notebook import tqdm
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}, Device: {device}")

PyTorch: 2.10.0+cu128, Device: cuda


##Download MedSAM2 Checkpoint

In [5]:
from huggingface_hub import hf_hub_download

# Create checkpoints directory
os.makedirs('checkpoints', exist_ok=True)

# Download the recommended MedSAM2 checkpoint (~160MB for tiny)
print("Downloading MedSAM2_latest.pt from HuggingFace...")
checkpoint_path = hf_hub_download(
    repo_id="wanglab/MedSAM2",
    filename="MedSAM2_latest.pt",
    local_dir="checkpoints",
    local_dir_use_symlinks=False
)
print(f"✓ Downloaded to: {checkpoint_path}")
print(f"  Size: {os.path.getsize(checkpoint_path)/1e6:.1f} MB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


MedSAM2_latest.pt:   0%|          | 0.00/156M [00:00<?, ?B/s]

✓ Downloaded to: checkpoints/MedSAM2_latest.pt
  Size: 156.0 MB


In [6]:
# Add MedSAM2 to path
sys.path.insert(0, 'MedSAM2')

# Import MedSAM2's build function
from sam2.build_sam import build_sam2

# MedSAM2 uses sam2.1_hiera_t512.yaml config (Tiny model, 512px input)
# This config is in MedSAM2/configs/
MODEL_CFG = "configs/sam2.1_hiera_t512.yaml"
CHECKPOINT = "checkpoints/MedSAM2_latest.pt"

# Build MedSAM2 model
print("Loading MedSAM2...")
medsam2_model = build_sam2(MODEL_CFG, CHECKPOINT, device=device)
print("✓ MedSAM2 loaded successfully!")

Loading MedSAM2...


/content/MedSAM2/sam2/modeling/sam/transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


✓ MedSAM2 loaded successfully!


##MedSAM2 Feature Extractor Wrapper

In [7]:
class MedSAM2FeatureExtractor(nn.Module):
    """
    Wrapper to extract multi-scale features from MedSAM2's image encoder.

    MedSAM2 uses SAM2.1-Tiny architecture with Hiera backbone.
    - Input: 512x512 images (MedSAM2's native size)
    - Output: Multi-scale feature maps from backbone FPN


    """

    def __init__(self, medsam2_model, freeze=True):
        super().__init__()
        self.image_encoder = medsam2_model.image_encoder

        # Freeze MedSAM2 encoder (preserve medical domain knowledge)
        if freeze:
            for param in self.image_encoder.parameters():
                param.requires_grad = False
            print("MedSAM2 encoder frozen (preserving medical fine-tuning)")

        # Channel dimensions will be detected on first forward pass
        self.ch = None
        self._initialized = False

    def forward(self, x):
        """
        Extract multi-scale features from MedSAM2.

        Args:
            x: Input tensor [B, 3, H, W] - will be resized to 512x512

        Returns:
            dict with keys 's1', 's2', 's3', 's4' containing feature maps
        """
        original_size = x.shape[2:]

        # MedSAM2 expects 512x512 input
        if original_size != (512, 512):
            x_resized = F.interpolate(x, size=(512, 512), mode='bilinear', align_corners=False)
        else:
            x_resized = x

        # Extract features (no gradients needed for frozen encoder)
        with torch.no_grad():
            encoder_output = self.image_encoder(x_resized)

        # Parse output structure
        # MedSAM2/SAM2 returns dict with 'backbone_fpn' and 'vision_features'
        if isinstance(encoder_output, dict):
            if 'backbone_fpn' in encoder_output:
                fpn = encoder_output['backbone_fpn']
                vf = encoder_output.get('vision_features', fpn[-1])

                features = {
                    's1': fpn[0] if len(fpn) > 0 else vf,
                    's2': fpn[1] if len(fpn) > 1 else vf,
                    's3': fpn[2] if len(fpn) > 2 else vf,
                    's4': vf
                }
            else:
                vf = encoder_output.get('vision_features', list(encoder_output.values())[0])
                features = {k: vf for k in ['s1', 's2', 's3', 's4']}
        else:
            # Fallback for tensor output
            features = {k: encoder_output for k in ['s1', 's2', 's3', 's4']}

        # Detect channel dimensions on first pass
        if not self._initialized:
            self.ch = {k: v.shape[1] for k, v in features.items()}
            print(f"MedSAM2 feature channels: {self.ch}")
            self._initialized = True

        return features


# Create feature extractor
medsam2_encoder = MedSAM2FeatureExtractor(medsam2_model, freeze=True).to(device)

# Test and detect channels
print("\nTesting MedSAM2 encoder...")
with torch.no_grad():
    test_input = torch.randn(1, 3, 512, 512).to(device)
    test_output = medsam2_encoder(test_input)
    print(f"Feature map shapes:")
    for k, v in test_output.items():
        print(f"  {k}: {v.shape}")

print(f"\n✓ MedSAM2 encoder ready with channels: {medsam2_encoder.ch}")

MedSAM2 encoder frozen (preserving medical fine-tuning)

Testing MedSAM2 encoder...
MedSAM2 feature channels: {'s1': 256, 's2': 256, 's3': 256, 's4': 256}
Feature map shapes:
  s1: torch.Size([1, 256, 128, 128])
  s2: torch.Size([1, 256, 64, 64])
  s3: torch.Size([1, 256, 32, 32])
  s4: torch.Size([1, 256, 32, 32])

✓ MedSAM2 encoder ready with channels: {'s1': 256, 's2': 256, 's3': 256, 's4': 256}


##Prepare Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/data

# Copy from Drive
src = '/content/drive/MyDrive/Retinal_Vessel_Data/'
if os.path.exists(src):
    !cp -r "$src"* /content/data/
    print("Copied from Drive")

# Decompress STARE masks
stare_labels = '/content/data/STARE/labels-ah'
if os.path.exists(stare_labels):
    for gz in glob(f'{stare_labels}/*.gz'):
        out = gz[:-3]
        if not os.path.exists(out):
            with gzip.open(gz,'rb') as fi, open(out,'wb') as fo:
                shutil.copyfileobj(fi, fo)
    print("STARE masks decompressed")

!ls /content/data/

Mounted at /content/drive


In [ ]:
class VesselDataset(Dataset):
    """Retinal vessel segmentation dataset with FOV mask support."""

    def __init__(self, img_paths, mask_paths, size=512, augment=False):
        self.imgs = img_paths
        self.masks = mask_paths
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        # Load image
        img = cv2.imread(self.imgs[i])
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img = np.array(Image.open(self.imgs[i]).convert('RGB'))

        # Load mask
        mask = cv2.imread(self.masks[i], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            mask = np.array(Image.open(self.masks[i]).convert('L'))

        # Create FOV mask (circular region of valid data)
        # FOV = anywhere image is not black padding
        fov = (img.sum(axis=2) > 30).astype(np.float32)

        # Resize
        img = cv2.resize(img, (self.size, self.size))
        mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        fov = cv2.resize(fov, (self.size, self.size), interpolation=cv2.INTER_NEAREST)

        # CLAHE enhancement on green channel (vessels most visible)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[:,:,0] = cv2.createCLAHE(2.0, (8,8)).apply(lab[:,:,0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

        # Augmentation
        if self.augment:
            if np.random.rand() > 0.5:
                img, mask, fov = np.fliplr(img).copy(), np.fliplr(mask).copy(), np.fliplr(fov).copy()
            if np.random.rand() > 0.5:
                img, mask, fov = np.flipud(img).copy(), np.flipud(mask).copy(), np.flipud(fov).copy()
            if np.random.rand() > 0.5:
                k = np.random.randint(1, 4)
                img, mask, fov = np.rot90(img, k).copy(), np.rot90(mask, k).copy(), np.rot90(fov, k).copy()

        # Normalize and convert to tensor
        img = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1)
        mask = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        fov = torch.from_numpy(fov).unsqueeze(0)

        return {
            'image': img,
            'mask': mask,
            'fov': fov,
            'name': os.path.basename(self.imgs[i])
        }

In [ ]:
def load_drive(d, sz=512):
    tr_i, tr_m = [], []
    for p in sorted(glob(f'{d}/training/images/*.tif')):
        n = os.path.basename(p).split('_')[0]
        m = glob(f'{d}/training/1st_manual/{n}_manual1.*')
        if m: tr_i.append(p); tr_m.append(m[0])

    # Check if test set has vessel masks
    te_i, te_m = [], []
    if os.path.exists(f'{d}/test/1st_manual'):
        for p in sorted(glob(f'{d}/test/images/*.tif')):
            n = os.path.basename(p).split('_')[0]
            m = glob(f'{d}/test/1st_manual/{n}_manual1.*')
            if m: te_i.append(p); te_m.append(m[0])
    else:
        # Fall back to 80/20 split of training
        k = int(len(tr_i) * 0.8)
        te_i, te_m = tr_i[k:], tr_m[k:]
        tr_i, tr_m = tr_i[:k], tr_m[:k]

    print(f"DRIVE: {len(tr_i)} train, {len(te_i)} test")
    return VesselDataset(tr_i, tr_m, sz, True), VesselDataset(te_i, te_m, sz, False)

def load_stare(d, sz=512):
    imgs, masks = [], []
    for p in sorted(glob(f'{d}/*.ppm')):
        n = os.path.splitext(os.path.basename(p))[0]
        m = f'{d}/labels-ah/{n}.ah.ppm'
        if os.path.exists(m): imgs.append(p); masks.append(m)
    k = min(10, len(imgs))
    print(f"STARE: {k} train, {len(imgs)-k} test")
    return VesselDataset(imgs[:k], masks[:k], sz, True), VesselDataset(imgs[k:], masks[k:], sz, False)

def load_chase(d, sz=512):
    imgs, masks = [], []
    for p in sorted(glob(f'{d}/Image_*.jpg')):
        n = os.path.splitext(os.path.basename(p))[0]
        m = f'{d}/{n}_1stHO.png'
        if os.path.exists(m): imgs.append(p); masks.append(m)
    k = min(20, len(imgs))
    print(f"CHASE_DB1: {k} train, {len(imgs)-k} test")
    return VesselDataset(imgs[:k], masks[:k], sz, True), VesselDataset(imgs[k:], masks[k:], sz, False)

def get_loaders(name, d, sz=512, bs=4):
    if 'DRIVE' in name.upper(): tr, te = load_drive(d, sz)
    elif 'STARE' in name.upper(): tr, te = load_stare(d, sz)
    else: tr, te = load_chase(d, sz)
    return DataLoader(tr, bs, True, num_workers=2), DataLoader(te, 1, False, num_workers=2)

##Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target, fov=None):
        # Apply FOV mask if provided
        if fov is not None:
            pred = pred * fov
            target = target * fov

        pred = pred.view(-1)
        target = target.view(-1)
        intersection = (pred * target).sum()
        return 1 - (2 * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)


class clDiceLoss(nn.Module):
    """Centerline Dice for tubular structures like vessels."""
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def soft_skel(self, x, iters=10):
        """Soft skeletonization via iterative erosion."""
        for _ in range(iters):
            x = torch.clamp(x - (-F.max_pool2d(-x, 3, 1, 1)), 0, 1)
        return x

    def forward(self, pred, target, fov=None):
        if fov is not None:
            pred = pred * fov
            target = target * fov

        pred_skel = self.soft_skel(pred)
        target_skel = self.soft_skel(target)

        tprec = ((pred_skel * target).sum() + self.smooth) / (pred_skel.sum() + self.smooth)
        tsens = ((target_skel * pred).sum() + self.smooth) / (target_skel.sum() + self.smooth)

        return 1 - (2 * tprec * tsens + self.smooth) / (tprec + tsens + self.smooth)


class CombinedLoss(nn.Module):
    """Combined Dice + clDice loss with FOV mask support."""
    def __init__(self, lambda_cldice=0.5):
        super().__init__()
        self.dice = DiceLoss()
        self.cldice = clDiceLoss()
        self.lam = lambda_cldice

    def forward(self, pred, target, fov=None):
        return self.dice(pred, target, fov) + self.lam * self.cldice(pred, target, fov)

##Model Architectures

In [ ]:
class ConvBlock(nn.Module):
    """Double convolution block with BatchNorm."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class FusionBlock(nn.Module):
    """
    Attention-based fusion of U-Net and MedSAM2 features.
    Uses learnable gate initialized to 0 (starts by ignoring MedSAM2).
    """
    def __init__(self, unet_ch, medsam_ch):
        super().__init__()
        # Project MedSAM2 features to U-Net channel size
        self.proj = nn.Sequential(
            nn.Conv2d(medsam_ch, unet_ch, 1),
            nn.BatchNorm2d(unet_ch),
            nn.ReLU(inplace=True)
        )
        # Learnable gate (starts at 0 = ignore MedSAM2 initially)
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, unet_feat, medsam_feat):
        # Resize MedSAM2 features to match U-Net spatial size
        if unet_feat.shape[2:] != medsam_feat.shape[2:]:
            medsam_feat = F.interpolate(medsam_feat, size=unet_feat.shape[2:],
                                        mode='bilinear', align_corners=False)

        # Project and add with learnable gate
        medsam_proj = self.proj(medsam_feat)
        gate = torch.sigmoid(self.gate)  # 0-1

        # Residual addition: output = unet + gate * (medsam - unet)
        # When gate=0: output = unet (ignore medsam)
        # When gate=1: output = medsam (replace with medsam)
        return unet_feat + gate * (medsam_proj - unet_feat)

In [ ]:
class BaselineUNet(nn.Module):
    """Standard U-Net without MedSAM2 features."""

    def __init__(self):
        super().__init__()
        # Encoder
        self.e1 = ConvBlock(3, 64)
        self.e2 = ConvBlock(64, 128)
        self.e3 = ConvBlock(128, 256)
        self.e4 = ConvBlock(256, 512)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bn = ConvBlock(512, 1024)

        # Decoder
        self.u4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.d4 = ConvBlock(1024, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.d3 = ConvBlock(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.d2 = ConvBlock(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.d1 = ConvBlock(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        # Encoder
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))

        # Bottleneck
        b = self.bn(self.pool(e4))

        # Decoder with skip connections
        d = self.d4(torch.cat([self.u4(b), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))

        return torch.sigmoid(self.out(d))

In [ ]:
class BottleneckFusionUNet(nn.Module):
    """U-Net with MedSAM2 feature fusion at bottleneck only."""

    def __init__(self, medsam_encoder):
        super().__init__()
        self.medsam = medsam_encoder

        # Encoder
        self.e1 = ConvBlock(3, 64)
        self.e2 = ConvBlock(64, 128)
        self.e3 = ConvBlock(128, 256)
        self.e4 = ConvBlock(256, 512)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bn = ConvBlock(512, 1024)

        # Fusion at bottleneck
        self.fuse = FusionBlock(1024, medsam_encoder.ch['s4'])

        # Decoder
        self.u4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.d4 = ConvBlock(1024, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.d3 = ConvBlock(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.d2 = ConvBlock(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.d1 = ConvBlock(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        # Get MedSAM2 features
        mf = self.medsam(x)

        # Encoder
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))

        # Bottleneck with MedSAM2 fusion
        b = self.bn(self.pool(e4))
        b = self.fuse(b, mf['s4'])

        # Decoder
        d = self.d4(torch.cat([self.u4(b), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))

        return torch.sigmoid(self.out(d))

In [ ]:
class MultiScaleFusionUNet(nn.Module):
    """U-Net with MedSAM2 feature fusion at ALL encoder levels."""

    def __init__(self, medsam_encoder):
        super().__init__()
        self.medsam = medsam_encoder
        ch = medsam_encoder.ch

        # Encoder
        self.e1 = ConvBlock(3, 64)
        self.e2 = ConvBlock(64, 128)
        self.e3 = ConvBlock(128, 256)
        self.e4 = ConvBlock(256, 512)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bn = ConvBlock(512, 1024)

        # Multi-scale fusion
        self.f1 = FusionBlock(64, ch['s1'])
        self.f2 = FusionBlock(128, ch['s2'])
        self.f3 = FusionBlock(256, ch['s3'])
        self.f4 = FusionBlock(512, ch['s4'])
        self.fbn = FusionBlock(1024, ch['s4'])

        # Decoder
        self.u4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.d4 = ConvBlock(1024, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.d3 = ConvBlock(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.d2 = ConvBlock(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.d1 = ConvBlock(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        # Get MedSAM2 features
        mf = self.medsam(x)

        # Encoder with multi-scale fusion
        e1 = self.f1(self.e1(x), mf['s1'])
        e2 = self.f2(self.e2(self.pool(e1)), mf['s2'])
        e3 = self.f3(self.e3(self.pool(e2)), mf['s3'])
        e4 = self.f4(self.e4(self.pool(e3)), mf['s4'])

        # Bottleneck with fusion
        b = self.fbn(self.bn(self.pool(e4)), mf['s4'])

        # Decoder
        d = self.d4(torch.cat([self.u4(b), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))

        return torch.sigmoid(self.out(d))

##Training Functions

In [ ]:
def compute_metrics(pred, target, fov=None):
    """Compute segmentation metrics with optional FOV mask."""
    # Apply FOV mask
    if fov is not None:
        pred = pred * fov
        target = target * fov

    # Binarize predictions
    p = (pred > 0.5).float()

    # Compute metrics
    tp = (p * target).sum()
    fp = (p * (1 - target)).sum()
    fn = ((1 - p) * target).sum()
    tn = ((1 - p) * (1 - target)).sum()

    eps = 1e-7
    return {
        'dice': (2 * tp / (2 * tp + fp + fn + eps)).item(),
        'iou': (tp / (tp + fp + fn + eps)).item(),
        'sensitivity': (tp / (tp + fn + eps)).item(),
        'specificity': (tn / (tn + fp + eps)).item()
    }


def train_model(model, train_loader, test_loader, epochs=100, lr=1e-4, model_name='model'):
    """Train model with FOV-aware loss."""
    criterion = CombinedLoss(lambda_cldice=0.5)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=10, factor=0.5)

    history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_dice': []}
    best_dice = 0

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
            img = batch['image'].to(device)
            mask = batch['mask'].to(device)
            fov = batch['fov'].to(device)

            optimizer.zero_grad()
            pred = model(img)
            loss = criterion(pred, mask, fov)  # FOV-aware loss
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # Evaluation
        model.eval()
        val_loss = 0
        metrics_list = []
        with torch.no_grad():
            for batch in test_loader:
                img = batch['image'].to(device)
                mask = batch['mask'].to(device)
                fov = batch['fov'].to(device)

                pred = model(img)
                val_loss += criterion(pred, mask, fov).item()
                metrics_list.append(compute_metrics(pred, mask, fov))

        val_loss /= len(test_loader)
        val_dice = np.mean([m['dice'] for m in metrics_list])

        # Record history
        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)

        scheduler.step(val_dice)

        # Print progress
        sen = np.mean([m['sensitivity'] for m in metrics_list])
        spe = np.mean([m['specificity'] for m in metrics_list])
        print(f"Epoch {epoch:3d} | Loss: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"Dice: {val_dice:.4f} | Sen: {sen:.4f} | Spe: {spe:.4f}")

        # Save best model
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), f'{model_name}_best.pth')
            print(f"         → New best! Dice: {best_dice:.4f}")

    # Final metrics
    final_metrics = {
        'dice': np.mean([m['dice'] for m in metrics_list]),
        'iou': np.mean([m['iou'] for m in metrics_list]),
        'sensitivity': np.mean([m['sensitivity'] for m in metrics_list]),
        'specificity': np.mean([m['specificity'] for m in metrics_list])
    }

    return history, final_metrics

##Run Experiments

In [ ]:
# ===== CONFIGURATION =====
DATASET = 'CHASE_DB1'  # Options: 'DRIVE', 'STARE', 'CHASE_DB1'
DATA_DIR = f'/content/data/{DATASET}'
EPOCHS = 100
BATCH_SIZE = 4
LR = 1e-4

print(f"Dataset: {DATASET}")
print(f"Epochs: {EPOCHS}")
print(f"Using MedSAM2 (medical fine-tuned)")

In [ ]:
# Load data
train_loader, test_loader = get_loaders(DATASET, DATA_DIR, 512, BATCH_SIZE)

# Storage for results
all_results = {}
all_histories = {}
all_models = {}

In [ ]:
# ========== BASELINE ==========
print("\n" + "="*60)
print("Training: BASELINE U-Net (no MedSAM2)")
print("="*60)

model_baseline = BaselineUNet().to(device)
history, metrics = train_model(model_baseline, train_loader, test_loader, EPOCHS, LR, 'baseline')

all_results['Baseline'] = metrics
all_histories['Baseline'] = history
all_models['Baseline'] = model_baseline

In [ ]:
# ========== BOTTLENECK FUSION ==========
print("\n" + "="*60)
print("Training: BOTTLENECK FUSION (MedSAM2 at bottleneck)")
print("="*60)

model_bottleneck = BottleneckFusionUNet(medsam2_encoder).to(device)
history, metrics = train_model(model_bottleneck, train_loader, test_loader, EPOCHS, LR, 'bottleneck')

all_results['Bottleneck'] = metrics
all_histories['Bottleneck'] = history
all_models['Bottleneck'] = model_bottleneck

In [ ]:
# ========== MULTI-SCALE FUSION ==========
print("\n" + "="*60)
print("Training: MULTI-SCALE FUSION (MedSAM2 at all levels)")
print("="*60)

model_multiscale = MultiScaleFusionUNet(medsam2_encoder).to(device)
history, metrics = train_model(model_multiscale, train_loader, test_loader, EPOCHS, LR, 'multiscale')

all_results['MultiScale'] = metrics
all_histories['MultiScale'] = history
all_models['MultiScale'] = model_multiscale

##Save Results

In [ ]:
# ========== 1. SAVE METRICS TABLE ==========
results_df = pd.DataFrame([
    {'Model': name, 'Dice': m['dice'], 'IoU': m['iou'],
     'Sensitivity': m['sensitivity'], 'Specificity': m['specificity']}
    for name, m in all_results.items()
])

csv_path = f'results_{DATASET}_MedSAM2.csv'
results_df.to_csv(csv_path, index=False)

print("\n" + "="*60)
print(f"RESULTS - {DATASET} (using MedSAM2)")
print("="*60)
print(results_df.to_string(index=False))
print(f"\n✓ Saved to: {csv_path}")

In [ ]:
# ========== 2. TRAINING CURVES ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Baseline': '#3B82F6', 'Bottleneck': '#F59E0B', 'MultiScale': '#10B981'}

# Loss
ax = axes[0]
for name, hist in all_histories.items():
    ax.plot(hist['epoch'], hist['train_loss'], '-', color=colors[name], label=f'{name} (train)', lw=2)
    ax.plot(hist['epoch'], hist['val_loss'], '--', color=colors[name], label=f'{name} (val)', lw=2, alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training & Validation Loss'); ax.legend(); ax.grid(True, alpha=0.3)

# Dice
ax = axes[1]
for name, hist in all_histories.items():
    ax.plot(hist['epoch'], hist['val_dice'], '-', color=colors[name], label=name, lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Dice Score')
ax.set_title('Validation Dice Score'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
curves_path = f'training_curves_{DATASET}_MedSAM2.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved to: {curves_path}")

In [ ]:
# ========== 3. EXAMPLE PREDICTIONS ==========
num_samples = 3
samples = [next(iter(test_loader)) for _ in range(num_samples)]

fig, axes = plt.subplots(num_samples, 5, figsize=(20, 4*num_samples))

for i, batch in enumerate(samples):
    img = batch['image'].to(device)
    mask = batch['mask'][0, 0].cpu().numpy()
    fov = batch['fov'][0, 0].cpu().numpy()

    # Input
    axes[i, 0].imshow(batch['image'][0].permute(1,2,0).numpy())
    axes[i, 0].set_title('Input'); axes[i, 0].axis('off')

    # Ground Truth
    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')

    # Model predictions (binarized)
    for j, (name, model) in enumerate(all_models.items()):
        model.eval()
        with torch.no_grad():
            pred = model(img)[0, 0].cpu().numpy()
        # Apply FOV mask and binarize
        pred = (pred > 0.5).astype(np.float32) * fov
        axes[i, j+2].imshow(pred, cmap='gray')
        axes[i, j+2].set_title(name); axes[i, j+2].axis('off')

plt.tight_layout()
preds_path = f'predictions_{DATASET}_MedSAM2.png'
plt.savefig(preds_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved to: {preds_path}")

In [ ]:
# ========== 4. DOWNLOAD FILES ==========
from google.colab import files

print("\n" + "="*60)
print("SUMMARY - MedSAM2 vs SAM2")
print("="*60)
print("Using MedSAM2 (fine-tuned on 455K+ medical images) should show:")
print("  - MultiScale > Bottleneck > Baseline")
print("  - Better thin vessel detection")
print("  - Cleaner boundaries")
print("="*60)

files.download(csv_path)
files.download(curves_path)
files.download(preds_path)